# Evidence Retrieval Experiment Framework

This notebook acts as the central experiment framework for the dissertation.

It does not perform exploratory analysis of AVeriTeC; this is covered in the separate data inspection notebook. Instead, this notebook:

1. defines the experimental and reproducibility settings;
2. loads the prepared claims and candidate evidence collection;
3. executes each retrieval configuration through a common interface;
4. evaluates the resulting rankings using the same metrics;
5. records the configuration and environment associated with each run;

Retrieval implementations are contained in separate Python modules so that the experimental procedure remains consistent across methods.

## 1. Imports

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import json
import os
import random
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

In [ ]:
from .data import load_claims, load_candidate_corpus

from src.retrieval.bm25 import BM25Retriever
from src.retrieval.dense import DenseRetriever
from src.retrieval.hybrid import HybridRetriever
from src.retrieval.reranker import CrossEncoderReranker

from src.evaluation import evaluate_retrieval

## 2. Experimental Configuration and Reproducibility

All variables capable of changing the experimental outcome are defined here.
Values should not be changed inside individual retrieval sections.

### 2.1. Main config cell

In [ ]:
EXPERIMENT = {
    "experiment_name": "initial_retriever_comparison",
    "seed": 67,

    "dataset": "AVeriTeC",
    "split": "dev",
    "dataset_revision": None,
    "data_root": os.environ.get("AVERITEC_ROOT"),

    "retrieve_unit": "passage", 
    "chunk_size": None, 
    "chunk_overlap": None, 
    "retrieve_k": 100,
    "evaluation_k": [1, 5, 10, 20, 50, 100],

    "dense_model": None,
    "dense_model_revision": None,
    "dense_batch_size": None,
    "dense_similarity": "dot_product",

    "hybrid_method": "rrf",
    "rrf_k": 60,
    "hybrid_alpha": None,

    "reranker_model": None,
    "reranker_model_revision": None,
    "reranker_candidates": 100,
    "reranker_output_k": 20,

    "device": "cuda" if torch.cuda.is_available() else "cpu",

}

# NOTE: None's to be filled in once determined.

### 2.1. Fix random seeds

In [ ]:
SEED = EXPERIMENT["seed"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seed: {SEED}")
print(f"Device: {EXPERIMENT['device']}")

# Enable if deterministic execution is required and supported
# torch.use_deterministic_algorithms(True)

### 2.2. Capture environment automatically

In [ ]:
def get_git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            text=True
        ).strip()
    except Exception:
        return None


ENVIRONMENT = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "git_commit": get_git_commit(),
}

pd.Series(ENVIRONMENT)

# NOTE: list not exhaustive, this is early expectations of what would be needed, should be updated later.

### 2.3. Required result validator

In [ ]:
REQUIRED_RESULT_COLUMNS = {
    "claim_id",
    "candidate_id",
    "rank",
    "score",
    "source_url",
    "text",
    "method",
}
# NOTE: to be adjusted once the design of the retrievers has been decided


def validate_results(results: pd.DataFrame):
    missing = REQUIRED_RESULT_COLUMNS - set(results.columns)

    if missing:
        raise ValueError(f"Retrieval output missing columns: {sorted(missing)}")

    return results

## 3. Load experimental data
Dataset structure and annotation characteristics are explored separately in
`01_averitec_data_inspection.ipynb`.  (NOTE: or whatever its called later)

Only the processed inputs required for retrieval are loaded here.

In [ ]:
claims = load_claims(root=Path(EXPERIMENT["data_root"]), split=EXPERIMENT["split"])
candidates = load_candidate_corpus(root=Path(EXPERIMENT["data_root"]), split=EXPERIMENT["split"])

print(f"Claims: {len(claims):,}")
print(f"Candidates: {len(candidates):,}")

# display(claims.head(2))
# display(candidates.head(2))

## 4. Load Retrievers

### 4.1. BM25 Retrieval

In [ ]:
bm25 = BM25Retriever(candidates=candidates)
bm25_results = bm25.retrieve(claims=claims, k=EXPERIMENT["retrieve_k"])
bm25_results = validate_results(bm25_results)
display(bm25_results.head())

# -- Evaluation: -- #

bm25_metrics = evaluate_retrieval(claims=claims, results=bm25_results, cutoffs=EXPERIMENT["evaluation_k"])
bm25_metrics

### 4.2. Dense Retrieval

In [ ]:
dense = DenseRetriever(candidates=candidates,
    model_name=EXPERIMENT["dense_model"],
    model_revision=EXPERIMENT["dense_model_revision"],
    device=EXPERIMENT["device"],
    batch_size=EXPERIMENT["dense_batch_size"],
)

dense_results = dense.retrieve(claims=claims, k=EXPERIMENT["retrieve_k"])
dense_results = validate_results(dense_results)

# -- Evaluation: -- #

dense_metrics = evaluate_retrieval(claims=claims, results=dense_results, cutoffs=EXPERIMENT["evaluation_k"])
dense_metrics

### 4.3. Hybrid Retrieval
This section essentially just consumes the existing results, opposed to regenerating the experiment results for both BM25 and DPR

In [ ]:
hybrid = HybridRetriever(method=EXPERIMENT["hybrid_method"], rrf_k=EXPERIMENT["rrf_k"], alpha=EXPERIMENT["hybrid_alpha"])

hybrid_results = hybrid.fuse(bm25_results=bm25_results, dense_results=dense_results, k=EXPERIMENT["retrieve_k"])
hybrid_results = validate_results(hybrid_results)

# -- Evaluation: -- #

hybrid_metrics = evaluate_retrieval(claims=claims, results=hybrid_results, cutoffs=EXPERIMENT["evaluation_k"])
hybrid_metrics

### 4.4. Reranking Retrieval
As with the hybrid retrieval, the reranker should consume candidate results from one of the first-stage systems

In [ ]:
reranker = CrossEncoderReranker(model_name=EXPERIMENT["reranker_model"], model_revision=EXPERIMENT["reranker_model_revision"], device=EXPERIMENT["device"])

reranked_results = reranker.rerank(claims=claims, candidates=hybrid_results, candidate_k=EXPERIMENT["reranker_candidates"], output_k=EXPERIMENT["reranker_output_k"])
reranked_results = validate_results(reranked_results)

# -- Evaluation: -- #

reranked_metrics = evaluate_retrieval(claims=claims, results=reranked_results, cutoffs=EXPERIMENT["evaluation_k"])
reranked_metrics

## 5. Central Comparison

### 5.1. Combine results into a single DataFrame

In [ ]:
comparison = pd.DataFrame({
    "BM25": bm25_metrics,
    "Dense": dense_metrics,
    "Hybrid": hybrid_metrics,
    "Hybrid + Reranker": reranked_metrics,
}).T

comparison

### 5.2. Save experiment manifest

In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIR = Path("outputs") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = {"experiment": EXPERIMENT, "environment": ENVIRONMENT}

with (OUTPUT_DIR / "manifest.json").open("w") as f:
    json.dump(manifest, f, indent=2, default=str)

comparison.to_csv(OUTPUT_DIR / "metrics.csv")

bm25_results.to_parquet(OUTPUT_DIR / "bm25_results.parquet", index=False)
dense_results.to_parquet(OUTPUT_DIR / "dense_results.parquet", index=False)
hybrid_results.to_parquet(OUTPUT_DIR / "hybrid_results.parquet", index=False)
reranked_results.to_parquet(OUTPUT_DIR / "reranked_results.parquet", index=False)